# Fine-tuning Llama 3.1 8B Instruct for bank KYC work

This notebook runs the whole pipeline for adapting **`meta-llama/Llama-3.1-8B-Instruct`** to answer
**KYC (Know Your Customer)** questions from bank staff:

| # | Step | What happens |
|---|------|--------------|
| 1 | Setup | Install libraries, log in to the Hugging Face Hub, check the GPU |
| 2 | Data | Load `kyc_basic.jsonl`, explore it, add paraphrased questions, build a separate evaluation set |
| 3 | Base model | Load Llama 3.1 8B Instruct in 4-bit (QLoRA) |
| 4 | Evaluation harness | Metrics (ROUGE-L, semantic similarity, keyword recall, length, latency) and behaviour tests |
| 5 | **Before** fine-tuning | Zero-shot, static few-shot and retrieval few-shot runs of the base model |
| 6 | Fine-tuning | QLoRA supervised fine-tuning with TRL `SFTTrainer` |
| 7 | **After** fine-tuning | Same tests on the fine-tuned model (zero-shot and few-shot) |
| 8 | Comparison | Tables, charts and side-by-side answers |
| 9 | Publish | Save the LoRA adapter, optionally merge it, push to the Hugging Face Hub |
| 10 | Inference | How to load and use the published model |

> **Hardware:** one GPU with 16 GB of VRAM or more (Colab T4 / L4 / A100, or similar). The model is loaded in 4-bit NF4, so it
> needs about 6 GB for inference and 10 to 14 GB for training.
>
> **Access:** Llama 3.1 is a gated model. Accept the licence at
> https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct and use a token with read access (write access if you want to push).

## 1. Setup

In [ ]:
# Run once in a fresh environment. On Colab, first clone the project so that requirements.txt and data/ exist:
#   !git clone https://github.com/YoucefLgr/Learning.git
#   %cd Learning/kyc-llm-finetuning
%pip install -q -r requirements.txt

In [ ]:
import os, json, time, random, re, gc, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed)
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from sentence_transformers import SentenceTransformer, util as st_util
from rouge_score import rouge_scorer

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 200)

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Configuration
Change these values to fit your setup. `HF_REPO_NAME` is the name of the Hub repository the adapter is pushed to (under your username).

In [ ]:
BASE_MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"
DATA_PATH       = "data/kyc_basic.jsonl"
OUTPUT_DIR      = Path("outputs")
ADAPTER_DIR     = OUTPUT_DIR / "llama31-kyc-lora"
HF_REPO_NAME    = "llama-3.1-8b-instruct-kyc-lora"
PUSH_TO_HUB     = False      # set True to publish the adapter (needs a write token)
MERGE_AND_PUSH  = False      # set True to also publish a merged full-weight model (~16 GB, needs lots of RAM)
EMBED_MODEL_ID  = "sentence-transformers/all-MiniLM-L6-v2"   # used for few-shot retrieval and scoring

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# bf16 on Ampere+ (A100, L4, RTX 30xx/40xx); fp16 on older cards such as the T4
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print("compute dtype:", COMPUTE_DTYPE)

### Hugging Face login
The token is read from the `HF_TOKEN` environment variable (or Colab secret) if set. Otherwise you get an interactive prompt.

In [ ]:
from huggingface_hub import login, whoami

token = os.environ.get("HF_TOKEN")
try:  # Colab secrets
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token)
else:
    login()  # interactive widget / prompt

HF_USERNAME = whoami()["name"]
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
print("Logged in as", HF_USERNAME, "->", HF_REPO_ID)

## 2. Data
### 2.1 Load and explore the raw data

In [ ]:
raw = pd.read_json(DATA_PATH, lines=True)
raw["id"] = range(len(raw))
raw["q_words"] = raw["instruction"].str.split().str.len()
raw["a_words"] = raw["output"].str.split().str.len()
print(f"{len(raw)} instruction/answer pairs")
display(raw[["id", "instruction", "output", "a_words"]])
display(raw[["q_words", "a_words"]].describe().T)

**What we learn:** there are only **15** Q&A pairs. They are short, factual answers about the KYC basics: onboarding steps, documents,
beneficial owners, PEPs, sanctions, CDD/EDD, risk, red flags, reporting and ongoing monitoring.

That shapes the rest of the notebook:
* **The dataset is far too small to hold anything out.** Removing even 3 items would remove facts the model is supposed to learn.
  Instead we train on **all 15 facts** and test **generalisation to new wording**. Each fact gets several training paraphrases,
  plus a *different* paraphrase that is kept for evaluation only.
* Behaviour tests (section 4.3) check the model on **new scenarios** that apply the facts, e.g. "the client matches the UN sanctions list".
* Overfitting is the main risk, so we use LoRA with few epochs and watch the evaluation loss.

### 2.2 Paraphrase augmentation (training) and a held-out paraphrase set (evaluation)
Each original question gets 3 hand-written paraphrases for training. None of the evaluation questions below appear in training.

In [ ]:
# id -> extra training phrasings of the original question (the original is always kept too)
TRAIN_PARAPHRASES = {
    0:  ["What is the meaning of the acronym KYC?", "KYC is short for what?", "Expand the abbreviation KYC."],
    1:  ["What is the purpose of KYC in banking?", "Why is KYC important for a bank?",
         "For what reasons do banks perform Know Your Customer checks?"],
    2:  ["Describe the KYC onboarding process.", "What does the KYC process look like for a new client?",
         "List the stages of customer onboarding under KYC."],
    3:  ["Which documents must a private person provide to open an account?",
         "What paperwork is required from an individual customer at onboarding?",
         "What ID and papers does a retail client need for a new account?"],
    4:  ["Which documents are required to onboard a corporate client?",
         "What does a business have to provide to open an account?",
         "List the KYC documents needed for a legal entity."],
    5:  ["Define beneficial owner in the KYC context.", "Who is the UBO of a company?",
         "What does ultimate beneficial owner mean?"],
    6:  ["What does PEP mean in banking?", "Define a Politically Exposed Person.",
         "Who is considered a PEP?"],
    7:  ["What does sanctions screening mean for a bank?", "Explain sanctions checks in KYC.",
         "Why do banks check clients against sanctions lists?"],
    8:  ["How is CDD different from EDD?", "Explain CDD versus EDD.",
         "What separates customer due diligence from enhanced due diligence?"],
    9:  ["Which clients are considered high risk?", "What are the high-risk factors in KYC?",
         "When is a customer classified as high risk?"],
    10: ["What if a customer does not provide the KYC documents?",
         "Can the bank open an account if the client won't give the required documents?",
         "What does the bank do when a client refuses to hand over KYC documents?"],
    11: ["Give an example of a KYC red flag.", "What counts as a warning sign when onboarding a client?",
         "What are red flags in customer onboarding?"],
    12: ["What is the right action when a bank employee suspects a client?",
         "How should staff handle a suspicious customer?",
         "I'm a bank employee and a client looks suspicious. What do I do?"],
    13: ["Is KYC a one-time process?", "Does KYC stop after the account is opened?",
         "Do banks repeat KYC checks after onboarding?"],
    14: ["Is it possible to open a bank account online?", "How does remote onboarding verify a client's identity?",
         "Can KYC be done digitally?"],
}

# Held-out evaluation set: one NEW phrasing per fact, the reference answer, and keyword groups.
# A keyword group counts as hit if ANY of its alternatives appears in the answer (case-insensitive).
EVAL_SET = [
    (0,  "Can you tell me what the letters KYC mean?", [["know your customer"]]),
    (1,  "What's the point of banks running KYC checks on their clients?",
         [["fraud"], ["money laundering"], ["terroris"], ["legal", "law", "regulat"]]),
    (2,  "Walk me through the KYC workflow for a new customer.",
         [["verify", "verification"], ["beneficial owner", "real owner", "ubo"], ["sanction"], ["pep", "politically exposed"],
          ["risk"], ["monitor"]]),
    (3,  "I'm a private person, what do I need to bring to open an account?",
         [["passport", "id card", "identity"], ["proof of address", "utility bill"]]),
    (4,  "What paperwork should a firm submit to get a business account?",
         [["registration", "register", "incorporation"], ["articles of association", "articles"], ["director"],
          ["beneficial owner", "ubo"]]),
    (5,  "Who counts as the ultimate beneficial owner of a company?",
         [["own", "owner"], ["control"], ["25"]]),
    (6,  "Explain what a politically exposed person is.",
         [["public"], ["minister", "judge", "official"], ["family", "relative", "associate"], ["extra", "enhanced", "additional"]]),
    (7,  "Why and how do banks screen clients against sanctions lists?",
         [["sanction"], ["eu", "un", "united nations", "ofac"], ["cannot", "can't", "not open", "prohibit", "refuse"]]),
    (8,  "How does enhanced due diligence differ from customer due diligence?",
         [["standard", "basic", "baseline"], ["high-risk", "high risk"], ["extra", "additional", "more"],
          ["source of", "where their money", "money comes from"]]),
    (9,  "Which factors would put a customer in the high-risk category?",
         [["pep", "politically exposed"], ["high-risk countr", "high risk countr", "jurisdiction"],
          ["complex"], ["cash"]]),
    (10, "A customer won't hand over the documents we asked for. What now?",
         [["cannot", "can't", "unable", "not be able"], ["not open", "refuse", "decline", "reject"]]),
    (11, "Give examples of warning signs when onboarding a new client.",
         [["fake", "forged", "false", "fraudulent"], ["inconsistent", "mismatch"], ["refus", "reluctan", "unwilling"],
          ["unclear", "unknown", "unexplained", "source of"]]),
    (12, "I think a client is suspicious. What should I do as a bank employee?",
         [["compliance"], ["report"], ["not tell", "not inform", "tipping off", "tip off", "not alert", "not disclose"]]),
    (13, "Is KYC a one-time check at account opening?",
         [["no"], ["monitor"], ["update", "review", "refresh"], ["high-risk", "high risk"]]),
    (14, "Can a customer be onboarded remotely without visiting a branch?",
         [["yes"], ["online", "remote", "digital"], ["selfie", "video"], ["id", "passport", "identity"]]),
]

train_rows = []
for _, r in raw.iterrows():
    for q in [r["instruction"]] + TRAIN_PARAPHRASES[r["id"]]:
        train_rows.append({"id": r["id"], "question": q, "answer": r["output"]})
train_df = pd.DataFrame(train_rows)

eval_df = pd.DataFrame([{"id": i, "question": q, "reference": raw.loc[i, "output"], "keywords": kw}
                        for i, q, kw in EVAL_SET])

# Leakage check: no evaluation question may appear in training
overlap = set(q.lower() for q in eval_df.question) & set(q.lower() for q in train_df.question)
assert not overlap, overlap
print(f"train examples: {len(train_df)} ({len(raw)} facts x {len(train_df)//len(raw)} phrasings)")
print(f"eval examples : {len(eval_df)} (held-out phrasings)")
display(train_df.sample(5, random_state=SEED))

### 2.3 Chat format
Llama 3.1 Instruct is a chat model, so every example is stored as a **conversational prompt/completion** pair (system + user → assistant).
TRL applies the Llama 3.1 chat template and, because the data is in prompt/completion format, computes the **loss on the answer tokens only**.
The same system prompt is used for training and for every evaluation run, so the comparisons are fair.

In [ ]:
SYSTEM_PROMPT = ("You are a KYC (Know Your Customer) compliance assistant for bank employees. "
                 "Answer questions about client onboarding, due diligence, sanctions, PEPs and AML "
                 "clearly, accurately and concisely.")

def to_prompt_completion(q, a):
    return {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
            "completion": [{"role": "assistant", "content": a}]}

train_ds = Dataset.from_list([to_prompt_completion(q, a) for q, a in zip(train_df.question, train_df.answer)]).shuffle(seed=SEED)
eval_ds  = Dataset.from_list([to_prompt_completion(q, a) for q, a in zip(eval_df.question, eval_df.reference)])
print(train_ds); print(json.dumps(train_ds[0], indent=2))

## 3. Load the base model (4-bit QLoRA setup)
The model is loaded **once** in 4-bit NF4 with double quantisation. That single object is used for the *before* tests and then fine-tuned with LoRA adapters.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# Llama 3.1 has no pad token; use its dedicated reserved padding token instead of EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = "<|finetune_right_pad_id|>"
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 4. Evaluation harness
### 4.1 Prompting strategies
* **Zero-shot:** system prompt + question only.
* **Static few-shot:** system prompt + **3 fixed examples** as earlier user/assistant turns + question. The examples teach the expected tone and length.
* **Retrieval few-shot (dynamic):** the **3 most similar training Q&As** (by embedding similarity) are used as examples.
  With a knowledge base this small, the retrieved examples usually contain the answer itself, so this behaves like a small *RAG* baseline.
  It is a strong reference point to compare fine-tuning against.

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL_ID, device="cuda" if torch.cuda.is_available() else "cpu")

# Few-shot pool = the ORIGINAL 15 Q&As (never the held-out eval phrasings)
POOL = raw[["id", "instruction", "output"]].rename(columns={"instruction": "question", "output": "answer"})
pool_emb = embedder.encode(POOL.question.tolist(), convert_to_tensor=True, normalize_embeddings=True)

STATIC_SHOT_IDS = [0, 6, 10]  # short definition, concept with detail, procedural answer

def _shots_to_messages(shots):
    msgs = []
    for _, s in shots.iterrows():
        msgs += [{"role": "user", "content": s.question}, {"role": "assistant", "content": s.answer}]
    return msgs

def build_zero_shot(q):
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]

def build_static_few_shot(q):
    shots = POOL.set_index("id").loc[STATIC_SHOT_IDS].reset_index()
    return [{"role": "system", "content": SYSTEM_PROMPT}] + _shots_to_messages(shots) + [{"role": "user", "content": q}]

def build_retrieval_few_shot(q, k=3):
    q_emb = embedder.encode(q, convert_to_tensor=True, normalize_embeddings=True)
    top = st_util.cos_sim(q_emb, pool_emb)[0].topk(k).indices.tolist()
    shots = POOL.iloc[top[::-1]]  # most similar example sits closest to the question
    return [{"role": "system", "content": SYSTEM_PROMPT}] + _shots_to_messages(shots) + [{"role": "user", "content": q}]

STRATEGIES = {"zero-shot": build_zero_shot,
              "few-shot (static)": build_static_few_shot,
              "few-shot (retrieval)": build_retrieval_few_shot}

for m in build_static_few_shot("What is a PEP?"):
    print(f"[{m['role']}] {m['content'][:90]}")

### 4.2 Generation and metrics
Decoding is **greedy** (`do_sample=False`), so runs are deterministic and comparable.

| Metric | What it measures |
|---|---|
| **ROUGE-L F1** | Word overlap with the reference answer (rewards matching the house style) |
| **Semantic similarity** | Cosine similarity of sentence embeddings, answer vs. reference (rewards the same meaning in different words) |
| **Keyword recall** | Share of the key KYC concepts (keyword groups) the answer mentions: a proxy for factual completeness |
| **Words** | Answer length (the reference answers are 6 to 40 words; staff need short answers) |
| **Latency** | Seconds per answer (longer answers cost more time and compute) |

In [ ]:
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

@torch.no_grad()
def generate(messages, max_new_tokens=256, mdl=None):
    mdl = mdl or model
    tokenizer.padding_side = "left"
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(mdl.device)
    out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       temperature=None, top_p=None, pad_token_id=tokenizer.pad_token_id)
    tokenizer.padding_side = "right"
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def has_term(text, term):
    # Word-boundary match. Short terms (<= 3 chars, e.g. "no", "un", "pep", "25") must be whole words
    # (optional plural "s"); longer terms match as prefixes, so "refus" matches "refuse" / "refusal".
    t = re.escape(term.lower())
    pattern = rf"\b{t}s?\b" if len(term) <= 3 else rf"\b{t}"
    return re.search(pattern, text.lower()) is not None

def keyword_recall(answer, groups):
    return float(np.mean([any(has_term(answer, alt) for alt in g) for g in groups]))

def evaluate(label, strategy, mdl=None):
    rows = []
    for _, r in eval_df.iterrows():
        t0 = time.perf_counter()
        ans = generate(STRATEGIES[strategy](r.question), mdl=mdl)
        dt = time.perf_counter() - t0
        rows.append({"model": label, "strategy": strategy, "id": r.id, "question": r.question, "answer": ans,
                     "rougeL": _rouge.score(r.reference, ans)["rougeL"].fmeasure,
                     "kw_recall": keyword_recall(ans, r.keywords),
                     "words": len(ans.split()), "latency_s": dt})
    df = pd.DataFrame(rows)
    ref_emb = embedder.encode(eval_df.reference.tolist(), convert_to_tensor=True, normalize_embeddings=True)
    ans_emb = embedder.encode(df.answer.tolist(), convert_to_tensor=True, normalize_embeddings=True)
    df["sem_sim"] = st_util.cos_sim(ans_emb, ref_emb).diagonal().cpu().numpy()
    return df

METRICS = ["rougeL", "sem_sim", "kw_recall", "words", "latency_s"]
def summarize(df):
    return df.groupby(["model", "strategy"], sort=False)[METRICS].mean().round(3)

### 4.3 Behaviour tests (scenario unit tests)
These are **pass/fail tests on new scenarios** that are not in the training data. The model has to *apply* the KYC rules, not repeat them.
A test passes if every `must` group is matched (any alternative) and no `must_not` phrase appears.
Keyword matching is a coarse proxy, so always read the printed answers as well.

In [ ]:
BEHAVIOUR_TESTS = [
    {"name": "no tipping-off",
     "q": "The client asked me whether we reported him to the authorities. Should I tell him?",
     "must": [["no", "must not", "should not", "cannot", "do not", "don't"], ["tipping", "tip off", "compliance", "confidential"]],
     "must_not": ["yes, you should tell", "you should inform the client"]},
    {"name": "sanctioned client",
     "q": "Our screening shows the applicant is on the UN sanctions list. Can we open the account?",
     "must": [["no", "cannot", "can't", "must not", "not open", "not be able"]],
     "must_not": ["yes, you can open"]},
    {"name": "missing documents",
     "q": "A company refuses to tell us who its beneficial owners are. Can we still onboard it?",
     "must": [["no", "cannot", "can't", "not open", "not be able", "unable"]],
     "must_not": ["yes, you can"]},
    {"name": "PEP relative",
     "q": "My new client is the brother of a government minister. Do I need to do anything special?",
     "must": [["pep", "politically exposed"], ["enhanced", "edd", "extra", "additional"]],
     "must_not": []},
    {"name": "UBO threshold",
     "q": "Someone holds 30% of a company's shares. Is that person a beneficial owner?",
     "must": [["yes"], ["25"]],
     "must_not": []},
    {"name": "ongoing monitoring",
     "q": "We onboarded a high-risk client last year. Are we done with KYC for them?",
     "must": [["no"], ["monitor", "review", "update"]],
     "must_not": []},
    {"name": "concise definition",
     "q": "In one sentence, what is CDD?",
     "must": [["customer due diligence"]],
     "must_not": [], "max_words": 60},
]

def run_behaviour_tests(label, mdl=None, strategy="zero-shot"):
    rows = []
    for t in BEHAVIOUR_TESTS:
        ans = generate(STRATEGIES[strategy](t["q"]), mdl=mdl)
        ok = (all(any(has_term(ans, alt) for alt in g) for g in t["must"])
              and not any(p in ans.lower() for p in t["must_not"]))
        if "max_words" in t:
            ok = ok and len(ans.split()) <= t["max_words"]
        rows.append({"model": label, "strategy": strategy, "test": t["name"], "passed": ok,
                     "words": len(ans.split()), "answer": ans})
    df = pd.DataFrame(rows)
    print(f"{label} / {strategy}: {df.passed.sum()}/{len(df)} behaviour tests passed")
    return df

## 5. Before fine-tuning: base model baselines
### 5.1 Quick smoke test

In [ ]:
for q in ["What does KYC stand for?", "What is a beneficial owner?"]:
    print("Q:", q); print("A:", generate(build_zero_shot(q)), "\n" + "-" * 80)

### 5.2 Zero-shot, static few-shot and retrieval few-shot on the held-out set

In [ ]:
results = []
for strat in STRATEGIES:
    df = evaluate("base", strat)
    results.append(df)
    print(f"done: base / {strat}")

base_results = pd.concat(results, ignore_index=True)
base_results.to_csv(OUTPUT_DIR / "eval_base.csv", index=False)
display(summarize(base_results))

In [ ]:
base_behaviour = pd.concat([run_behaviour_tests("base", strategy=s) for s in ["zero-shot", "few-shot (static)"]],
                           ignore_index=True)
display(base_behaviour[["strategy", "test", "passed", "words", "answer"]])

**What to look for:** the base model usually knows KYC well, and its zero-shot keyword recall is often good. But its answers are **long**
(bullet lists, disclaimers, often 150+ words where the reference uses 20 to 40), so ROUGE-L is low and latency is high.
Few-shot prompts shorten the answers and move them towards the house style. Retrieval few-shot scores highest, because the prompt
already contains the answer. It also costs about 3 extra Q&A pairs of context tokens on every call and needs a retrieval index.

## 6. Fine-tuning with QLoRA
### 6.1 LoRA and training configuration
* **LoRA** on all attention and MLP projections, rank 16, alpha 32, dropout 0.05. That is about 42M trainable parameters, **about 0.5%** of the model.
* **Completion-only loss:** the model learns to produce answers, not to repeat prompts.
* **3 epochs** at LR 2e-4 with a cosine schedule. With 60 short examples, more epochs quickly lead to memorising the text word for word.
* Evaluation loss on the held-out phrasings is computed every epoch. The best checkpoint is restored at the end.

In [ ]:
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,          # float < 1 = fraction of total steps (transformers >= 5)
    weight_decay=0.0,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True,
    max_length=512,
    packing=False,
    completion_only_loss=True,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

Sanity check: only the answer tokens should count towards the loss (label ≠ -100).

In [ ]:
sample = trainer.train_dataset[0]
batch = trainer.data_collator([sample])
labels = batch["labels"][0]
print("FULL TEXT :", tokenizer.decode(batch["input_ids"][0]))
print("\nTRAINED ON:", tokenizer.decode(batch["input_ids"][0][labels != -100]))

### 6.2 Train

In [ ]:
t0 = time.time()
train_result = trainer.train()
print(f"Training finished in {(time.time() - t0) / 60:.1f} min")
print(train_result.metrics)

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
tr = hist.dropna(subset=["loss"]) if "loss" in hist else pd.DataFrame()
ev = hist.dropna(subset=["eval_loss"]) if "eval_loss" in hist else pd.DataFrame()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tr["step"], tr["loss"], color="#2a78d6", lw=2, label="train loss")
ax.plot(ev["step"], ev["eval_loss"], color="#eb6834", lw=2, marker="o", ms=8, label="eval loss (held-out phrasings)")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title("QLoRA training curve", loc="left")
ax.grid(alpha=0.2); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "training_curve.png", dpi=150); plt.show()

Train loss should fall steadily and eval loss should fall and then flatten. If eval loss **rises** while train loss heads to 0,
the model is memorising: lower `num_train_epochs` or the learning rate. `load_best_model_at_end` already keeps the best epoch.

In [ ]:
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
ft_model = trainer.model
ft_model.eval(); ft_model.config.use_cache = True
print("Adapter saved to", ADAPTER_DIR)

## 7. After fine-tuning: same tests on the fine-tuned model

In [ ]:
for q in ["What does KYC stand for?", "What is a beneficial owner?"]:
    print("Q:", q); print("A:", generate(build_zero_shot(q), mdl=ft_model), "\n" + "-" * 80)

In [ ]:
ft_results = pd.concat([evaluate("fine-tuned", s, mdl=ft_model) for s in STRATEGIES], ignore_index=True)
ft_results.to_csv(OUTPUT_DIR / "eval_finetuned.csv", index=False)
display(summarize(ft_results))

In [ ]:
ft_behaviour = pd.concat([run_behaviour_tests("fine-tuned", mdl=ft_model, strategy=s)
                          for s in ["zero-shot", "few-shot (static)"]], ignore_index=True)
display(ft_behaviour[["strategy", "test", "passed", "words", "answer"]])

### 7.1 Regression check: general abilities
Fine-tuning on a tiny dataset can damage general behaviour (*catastrophic forgetting*). PEFT's `disable_adapter()` gives us the base
model from the same object, so both versions can be compared on off-topic prompts.

In [ ]:
GENERAL_PROMPTS = ["What is 17 * 23?", "Write a haiku about autumn.", "Translate 'good morning' into French."]
for q in GENERAL_PROMPTS:
    msgs = [{"role": "user", "content": q}]
    with ft_model.disable_adapter():
        base_ans = generate(msgs, max_new_tokens=80, mdl=ft_model)
    ft_ans = generate(msgs, max_new_tokens=80, mdl=ft_model)
    print(f"Q: {q}\n  base      : {base_ans}\n  fine-tuned: {ft_ans}\n")

## 8. Before vs after comparison

In [ ]:
all_results = pd.concat([base_results, ft_results], ignore_index=True)
all_results.to_csv(OUTPUT_DIR / "eval_all.csv", index=False)
summary = summarize(all_results)
summary.to_csv(OUTPUT_DIR / "summary.csv")
display(summary.style.format("{:.3f}")
        .highlight_max(subset=["rougeL", "sem_sim", "kw_recall"], color="#d6e7fb")
        .highlight_min(subset=["words", "latency_s"], color="#d6e7fb"))

In [ ]:
# Grouped bars: one group per quality metric, one bar per model x strategy configuration
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]  # fixed categorical order
cfg = summary.reset_index()
cfg["config"] = cfg["model"] + " | " + cfg["strategy"]
quality = ["rougeL", "sem_sim", "kw_recall"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [3, 1.2]})
w = 0.8 / len(cfg); x = np.arange(len(quality))
for i, row in cfg.iterrows():
    bars = axes[0].bar(x + i * w - 0.4 + w / 2, row[quality].astype(float), w * 0.92,
                       color=PALETTE[i % len(PALETTE)], label=row["config"])
    axes[0].bar_label(bars, fmt="%.2f", fontsize=7, color="#52514e", padding=2)
axes[0].set_xticks(x, ["ROUGE-L", "Semantic similarity", "Keyword recall"])
axes[0].set_ylim(0, 1.08); axes[0].set_title("Answer quality (higher is better)", loc="left")
axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper left")

bars = axes[1].barh(cfg["config"], cfg["words"], color=[PALETTE[i % len(PALETTE)] for i in range(len(cfg))], height=0.6)
axes[1].bar_label(bars, fmt="%.0f", fontsize=8, color="#52514e", padding=3)
axes[1].axvline(raw.a_words.mean(), color="#52514e", ls="--", lw=1)
axes[1].text(raw.a_words.mean(), -0.9, " reference avg", fontsize=8, color="#52514e")
axes[1].invert_yaxis(); axes[1].set_title("Avg answer length (words)", loc="left")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y" if ax is axes[0] else "x", alpha=0.2)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "comparison.png", dpi=150); plt.show()

In [ ]:
behaviour = pd.concat([base_behaviour, ft_behaviour], ignore_index=True)
behaviour.to_csv(OUTPUT_DIR / "behaviour_tests.csv", index=False)
display(behaviour.pivot_table(index="test", columns=["model", "strategy"], values="passed", aggfunc="first")
        .replace({True: "PASS", False: "FAIL"}))
display(behaviour.groupby(["model", "strategy"])["passed"].agg(["sum", "count"]).rename(columns={"sum": "passed", "count": "total"}))

### 8.1 Side-by-side answers (zero-shot)

In [ ]:
side = (all_results[all_results.strategy == "zero-shot"]
        .pivot(index="question", columns="model", values="answer")
        .join(eval_df.set_index("question")["reference"]))
for q, r in side.iterrows():
    print(f"Q: {q}\n  REFERENCE : {r['reference']}")
    print("  BASE      : " + textwrap.shorten(r["base"], 400))
    print("  FINE-TUNED: " + r["fine-tuned"] + "\n")

### 8.2 Reading the results
The exact numbers depend on GPU, library versions and seed. The pattern to expect:

1. **Style and conciseness improve the most.** The fine-tuned model answers in the short, direct style of the reference answers, with no
   long bullet lists or disclaimers. ROUGE-L and semantic similarity go up, and answer length and latency drop sharply (often 3 to 5 times fewer tokens).
2. **Zero-shot fine-tuned vs. few-shot base:** fine-tuning gets few-shot-level behaviour **without spending context tokens on examples**.
   Every call is shorter and cheaper.
3. **Retrieval few-shot is a strong competitor on facts.** When the answer is in the prompt, the base model copies it. For a real bank the
   best system is usually **fine-tuning for behaviour and style plus retrieval for facts and policies**, because policies change and should not be baked into weights.
4. **Behaviour tests** show whether the model applies the rules to new scenarios (tipping-off, sanctions hits, missing UBO information).
5. **The general-ability check** should show that arithmetic, writing and translation still work. With only 60 examples and LoRA, forgetting is usually minimal.

**Limitations:** 15 source facts is a teaching-sized dataset. Keyword metrics are crude, and the evaluation set only measures paraphrase
generalisation. For production use: build hundreds to thousands of reviewed Q&As from the bank's own KYC/AML policies, cover
jurisdiction-specific rules (EU AMLD, FATF, local regulators), add an LLM-as-judge or expert review, and keep a human in the loop for every decision.

## 9. Save and publish to the Hugging Face Hub
The **LoRA adapter** is published (about 170 MB) together with a model card. You can also publish a merged full-weight model.
Merging needs the base model in 16-bit (about 16 GB of CPU RAM or VRAM).

In [ ]:
metrics_md = summary.reset_index().to_markdown(index=False)
MODEL_CARD = f'''---
base_model: {BASE_MODEL_ID}
library_name: peft
license: llama3.1
language: [en]
tags: [kyc, aml, compliance, banking, qlora, lora, trl, sft]
pipeline_tag: text-generation
---

# {HF_REPO_NAME}

QLoRA adapter for **{BASE_MODEL_ID}**, fine-tuned to answer bank **KYC (Know Your Customer)** questions
(onboarding steps, documents, beneficial owners, PEPs, sanctions screening, CDD/EDD, risk, red flags, reporting,
ongoing monitoring) clearly and concisely.

## Training
* Data: 15 KYC Q&A pairs x 4 phrasings = {len(train_df)} examples (conversational prompt/completion, completion-only loss)
* Method: QLoRA (4-bit NF4), LoRA r=16, alpha=32, dropout=0.05, all linear projections
* 3 epochs, LR 2e-4 cosine, effective batch size 8

## Evaluation (held-out paraphrased questions, greedy decoding)
{metrics_md}

## Usage
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
tok = AutoTokenizer.from_pretrained("{BASE_MODEL_ID}")
base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL_ID}", dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, "{HF_REPO_ID}")
```

## Limitations
Educational project trained on a very small dataset. Not legal or compliance advice. Do not use it to make
onboarding decisions without human review and the institution's own policies.
'''
(ADAPTER_DIR / "README.md").write_text(MODEL_CARD)
print(MODEL_CARD[:1200])

In [ ]:
if PUSH_TO_HUB:
    ft_model.push_to_hub(HF_REPO_ID, private=True, commit_message="KYC QLoRA adapter")
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    from huggingface_hub import HfApi
    HfApi().upload_file(path_or_fileobj=str(ADAPTER_DIR / "README.md"), path_in_repo="README.md", repo_id=HF_REPO_ID)
    print("Adapter pushed: https://huggingface.co/" + HF_REPO_ID)
else:
    print("PUSH_TO_HUB is False: adapter kept locally in", ADAPTER_DIR)

In [ ]:
if MERGE_AND_PUSH:
    # Free the 4-bit training model, reload the base in 16-bit, merge the adapter weights into it
    del trainer, model, ft_model; gc.collect(); torch.cuda.empty_cache()
    base16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=COMPUTE_DTYPE, device_map="auto")
    merged = PeftModel.from_pretrained(base16, str(ADAPTER_DIR)).merge_and_unload()
    merged_repo = HF_REPO_ID.replace("-lora", "") + "-merged"
    merged.push_to_hub(merged_repo, private=True, safe_serialization=True, max_shard_size="5GB")
    tokenizer.push_to_hub(merged_repo, private=True)
    print("Merged model pushed: https://huggingface.co/" + merged_repo)

## 10. Inference with the published model
This is how a downstream application (a compliance chatbot or an internal help desk tool) loads the adapter from the Hub, or from the local folder.

In [ ]:
def load_kyc_model(adapter=HF_REPO_ID if PUSH_TO_HUB else str(ADAPTER_DIR)):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config,
                                                dtype=COMPUTE_DTYPE, device_map="auto")
    return tok, PeftModel.from_pretrained(base, adapter).eval()

def ask_kyc(question, mdl):
    return generate(build_zero_shot(question), mdl=mdl)

# Reuse the in-memory model if it still exists, otherwise load from disk / Hub
kyc_model = ft_model if "ft_model" in globals() else load_kyc_model()[1]
for q in ["A new corporate client has three shareholders with 40%, 35% and 25%. Who are the beneficial owners?",
          "What extra checks do we run for a high-risk client?",
          "Can a customer open an account with only a selfie?"]:
    print("Q:", q); print("A:", ask_kyc(q, kyc_model), "\n")

## Summary
* We loaded **Llama 3.1 8B Instruct** in 4-bit, built a KYC evaluation harness (reference metrics and scenario tests), and measured
  **zero-shot**, **static few-shot** and **retrieval few-shot** baselines **before** training.
* We fine-tuned with **QLoRA and TRL** on the KYC dataset (with paraphrase augmentation) and ran the **same tests after** training,
  including a general-ability regression check.
* We published the adapter and a model card to the **Hugging Face Hub**.

**Next steps:** grow the dataset with real policy-derived Q&As, add retrieval over internal KYC procedures (RAG), evaluate with an
LLM-as-judge and compliance experts, and consider DPO on expert preference pairs to strengthen refusal and escalation behaviour.